# Suite LOG — The logical layout

`Logical` defines a diagram's logical layout: points, vectors, rotations and shapes, with no coordinates, and resolves the compass of a point or a shape: mbse-basis [CAP-237: logical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-237), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import { Builders, Validators } from "@mbse/schemas/Framework";

import { Logical } from "../src/Framework/index.js";

function assert(condition: unknown, message = "assertion failed"): asserts condition {
  if (!condition) throw new Error(`AssertionError: ${message}`);
}

const NAMES = ["Logical.OfRotation", "Logical.OfPoint", "Logical.Vector", "Logical.OfShape", "Logical.Origin",
  "Logical.Borders", "Logical.Extents", "Logical.Containment", "Logical.Adjacency"];
const store: any = new Builders.Store();
store.transaction(Logical.install);
const L = store.Logical;
const problems = (schema: any, root: any): string[] => Validators.Validate(store).Reachable(schema, root);

function point(): any {
  return L.OfPoint((p: any) => p);
}

function turn(degrees: number): (r: any) => any {
  return (r: any) => r.degrees(degrees);
}

function raises(message: string, block: () => unknown): void {
  try {
    block();
  } catch (error) {
    assert(error instanceof Error && error.name === "ValueError", String(error));
    assert(error.message === message, error.message);
    return;
  }
  throw new Error(`AssertionError: expected ValueError: ${message}`);
}

<a id="LOG-01"></a>
## LOG-01 · The install defines the logical layout's schemas, each valid, and run again writes nothing

In [ ]:
const schemas = NAMES.map((name) => store.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
store.transaction(Logical.install);
assert(NAMES.every((name, i) => store.registered(name) === schemas[i]));

<a id="LOG-02"></a>
## LOG-02 · A shape has one origin, and border and extent points of its own

In [ ]:
let o: any, b: any, e: any, shape: any, other: any;
store.transaction(() => {
  [o, b, e] = [point(), point(), point()];
  shape = L.OfShape((s: any) => s
    .origin((a: any) => a.insert((x: any) => x.point(o)))
    .borders((a: any) => a.insert((x: any) => x.point(b)))
    .extents((a: any) => a.insert((x: any) => x.point(e))));
  L.OfPoint(o, (p: any) => p.outgoing((a: any) => a
    .insert((x: any) => x.end(b).rotation(turn(90)))
    .insert((x: any) => x.end(e).rotation(turn(180)))));
});
assert([...shape.origin].map((x: any) => x.point).join() === [o].join() && [...shape.origin][0].point === o);
assert([...shape.borders][0].point === b && [...shape.extents][0].point === e);
assert(problems(L.OfShape, shape).length === 0);

store.transaction(() => L.OfShape(shape, (s: any) => s.origin((a: any) => a.insert((x: any) => x.point(point())))));
assert(JSON.stringify(problems(L.OfShape, shape)) ===
  JSON.stringify(["unique(point) violated: entries agreeing on ['shape'] differ on ['point']"]));
store.transaction(() => {
  L.OfShape(shape, (s: any) => s.origin((a: any) => a.clear().insert((x: any) => x.point(o))));
  other = L.OfShape((s: any) => s.borders((a: any) => a.insert((x: any) => x.point(b))));
});
assert(JSON.stringify(problems(L.OfShape, other)) ===
  JSON.stringify(["unique(shape) violated: entries agreeing on ['point'] differ on ['shape']"]));
store.transaction(() => L.OfShape(other, (s: any) => s.borders((a: any) => a.clear())));
assert(problems(L.OfShape, shape).length === 0);

<a id="LOG-03"></a>
## LOG-03 · A vector has one rotation per start and end

In [ ]:
let start: any, end: any;
store.transaction(() => {
  [start, end] = [point(), point()];
  L.OfPoint(start, (p: any) => p.outgoing((a: any) => a.insert((x: any) => x.end(end).rotation(turn(45)))));
});
const [vector]: any[] = [...end.incoming];
assert(vector.start === start && vector.end === end && vector.rotation.degrees.get() === 45);
assert(problems(L.OfPoint, start).length === 0);
store.transaction(() =>
  L.OfPoint(start, (p: any) => p.outgoing((a: any) => a.insert((x: any) => x.end(end).rotation(turn(135))))));
assert(JSON.stringify(problems(L.OfPoint, start)) ===
  JSON.stringify(["unique(rotation) violated: entries agreeing on ['end', 'start'] differ on ['rotation']"]));

<a id="LOG-04"></a>
## LOG-04 · A shape has one container, and adjacencies to its siblings by rotation

In [ ]:
let box: any, a: any, c: any;
store.transaction(() => {
  [box, a, c] = [L.OfShape((s: any) => s), L.OfShape((s: any) => s), L.OfShape((s: any) => s)];
  for (const child of [a, c]) L.OfShape(child, (s: any) => s.container((x: any) => x.insert((e: any) => e.container(box))));
  L.OfShape(a, (s: any) => s.adjacent_to((x: any) => x.insert((e: any) => e.to_shape(c).rotation(turn(90)))));
});
const children = [...box.children].map((e: any) => e.child);
assert(children.length === 2 && children[0] === a && children[1] === c && problems(L.OfShape, box).length === 0);
const [adjacency]: any[] = [...c.adjacent_from];
assert(adjacency.from_shape === a && adjacency.rotation.degrees.get() === 90);
store.transaction(() => L.OfShape(c, (s: any) => s.container((x: any) => x.insert((e: any) => e.container(a)))));
assert(JSON.stringify(problems(L.OfShape, c)) ===
  JSON.stringify(["unique(container) violated: entries agreeing on ['child'] differ on ['container']"]));

<a id="LOG-05"></a>
## LOG-05 · A compass: a shape's and its origin's north, any other point's its preceding vector's start's, turned

In [ ]:
let p: any, q: any, r: any;
store.transaction(() => {
  [p, q, r] = [point(), point(), point()];
  L.OfPoint(b, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(p).rotation(turn(300)))));
  L.OfPoint(p, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(q).rotation(turn(-90)))));
  L.OfPoint(e, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(r).rotation(turn(0.5)))));
});
assert(Logical.compass(shape) === 0 && Logical.compass(o) === 0);
assert(Logical.compass(b) === 90 && Logical.compass(e) === 180);
assert(Logical.compass(p) === 30 && Logical.compass(q) === 300 && Logical.compass(r) === 180.5);

<a id="LOG-06"></a>
## LOG-06 · A compass the rules leave undefined is refused

In [ ]:
let lone: any, s: any, t: any, u: any, free: any, empty: any;
store.transaction(() => {
  [lone, s, t, u, free] = [point(), point(), point(), point(), point()];
  empty = L.OfShape((x: any) => x);
  L.OfPoint(lone, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(s)).insert((v: any) => v.end(t).rotation(turn(1)))));
  L.OfPoint(o, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(t).rotation(turn(2)))));
  L.OfPoint(t, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(u).rotation(turn(3)))));
  L.OfPoint(u, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(lone).rotation(turn(4)))));
});
raises("a shape without an origin has no compass", () => Logical.compass(empty));
raises("a point with 0 preceding vectors, not one, has no compass", () => Logical.compass(free));
raises("a point with 2 preceding vectors, not one, has no compass", () => Logical.compass(t));
raises("a point whose preceding vector has no rotation has no compass", () => Logical.compass(s));
let m: any, n: any;
store.transaction(() => {
  [m, n] = [point(), point()];
  L.OfPoint(m, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(n).rotation(turn(5)))));
  L.OfPoint(n, (x: any) => x.outgoing((a: any) => a.insert((v: any) => v.end(m).rotation(turn(6)))));
});
raises("a point on a cycle of vectors has no compass", () => Logical.compass(m));